# 1. Import Libraries

In [1]:
import sys
from pathlib import Path

# Thêm thư mục gốc (project root) vào Python path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error, root_mean_squared_error

import numpy as np

from utils.features import build_features

import optuna

# 2. Load Dataset

In [2]:
df_train, df_val = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = None,
                                pls_fit_years = [1]
                                   )

In [3]:
features = [
    'Year', 'Month', 'Day', 'Hour',
    'Combined_Temp',
    'Combined_GHI_1', 'Combined_GHI_2',
    'sin_day_k1', 'cos_day_k1',
    'sin_day_k2', 'cos_day_k2',
    'sin_week_k1', 'cos_week_k1',
]
target = 'Load'

# 3. Split Data

In [4]:
# Year 1 → TRAIN
train_mask = df_train['Year'] == 1

# Year 2 → VALIDATION
val_mask = df_val['Year'] == 2

# Year 3 → TEST
# Testing.xlsx is assumed to contain Year 3

X_train = df_train.loc[train_mask, features]
y_train = df_train.loc[train_mask, target]

X_val = df_val.loc[val_mask, features]
y_val = df_val.loc[val_mask, target]

print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")

Dataset sizes:
Train (Year 1): 8,784
Validation (Year 2): 8,760


# 4. Train on year 1, evaluate on year 2

In [5]:
tscv = TimeSeriesSplit(n_splits=4)

In [6]:
def objective(trial):

    params = {
        "objective": "reg:squarederror",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            1500
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.2,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            10,
            log=True
        ),

        "random_state": 42,
        "n_jobs": -1
    }

    fold_rmse = []

    # ============================================
    # Time-series CV trên YEAR 1
    # ============================================

    for train_idx, val_idx in tscv.split(X_train):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)

        model.fit(
            X_fold_train,
            y_fold_train
        )

        y_pred = model.predict(
            X_fold_val
        )

        rmse = root_mean_squared_error(
            y_fold_val,
            y_pred
        )

        fold_rmse.append(rmse)

    # Optuna minimize mean CV RMSE
    return np.mean(fold_rmse)

In [7]:
study = optuna.create_study(
    direction="minimize",
    study_name="xgb_year1_timeseries_cv"
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)

[I 2026-10-02 18:44:38,875] A new study created in memory with name: xgb_year1_timeseries_cv


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-10-02 18:44:42,211] Trial 0 finished with value: 298.40657806396484 and parameters: {'n_estimators': 1047, 'learning_rate': 0.0865965199582373, 'max_depth': 8, 'min_child_weight': 5, 'subsample': 0.8222118730347296, 'colsample_bytree': 0.7170433035970051, 'gamma': 3.916299657924619, 'reg_alpha': 0.17534762675261947, 'reg_lambda': 0.011241623262454207}. Best is trial 0 with value: 298.40657806396484.
[I 2026-10-02 18:44:44,619] Trial 1 finished with value: 279.416690826416 and parameters: {'n_estimators': 1293, 'learning_rate': 0.011551587496466223, 'max_depth': 5, 'min_child_weight': 3, 'subsample': 0.887232058588332, 'colsample_bytree': 0.7943107250408034, 'gamma': 4.016646234326118, 'reg_alpha': 0.0016412894626760249, 'reg_lambda': 0.006077187730249041}. Best is trial 1 with value: 279.416690826416.
[I 2026-10-02 18:44:48,758] Trial 2 finished with value: 287.10847091674805 and parameters: {'n_estimators': 1398, 'learning_rate': 0.024926820123248806, 'max_depth': 8, 'min_chil

In [8]:
print("Best CV RMSE:")
print(study.best_value)

print("\nBest parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best CV RMSE:
258.60409927368164

Best parameters:
n_estimators: 1064
learning_rate: 0.06292531041929814
max_depth: 3
min_child_weight: 7
subsample: 0.6424403066647434
colsample_bytree: 0.648748536282336
gamma: 4.180675000275074
reg_alpha: 3.6851039306954485e-06
reg_lambda: 0.0016090619993514859


In [9]:
best_params = study.best_params

seeds = [0, 1, 2, 3, 42]

results = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    y_pred = model.predict(X_val)

    results.append({
        "Seed": seed,
        "RMSE": root_mean_squared_error(y_val, y_pred),
        "MSE": mean_squared_error(y_val, y_pred),
        "MAE": mean_absolute_error(y_val, y_pred),
        "MAPE": mean_absolute_percentage_error(y_val, y_pred),
        "R2": r2_score(y_val, y_pred)
    })

stability_df = pd.DataFrame(results)

stability_summary = stability_df.drop(
    columns="Seed"
).agg(["mean", "std", "min", "max"]).T

print("=== Results by seed ===")
display(stability_df)

print("\n=== Stability summary ===")
display(stability_summary)

=== Results by seed ===


,Seed,RMSE,MSE,MAE,MAPE,R2
0,0,159.434525,25419.365234,115.260513,0.054273,0.846138
1,1,160.490601,25757.234375,115.878036,0.054505,0.844093
2,2,160.900787,25889.064453,116.208023,0.054677,0.843295
3,3,159.959549,25587.056641,115.620773,0.054448,0.845123
4,42,162.288437,26337.537109,117.828964,0.055498,0.840580



=== Stability summary ===


,mean,std,min,max
RMSE,160.614780,1.086356,159.434525,162.288437
MSE,25798.051563,349.612727,25419.365234,26337.537109
MAE,116.159262,0.995878,115.260513,117.828964
MAPE,0.054680,0.000480,0.054273,0.055498
R2,0.843846,0.002116,0.840580,0.846138


In [10]:
stability_df.to_csv("../report/seed_daily_second_harmonic_val_report.csv", index = False)
stability_summary.to_csv("../report/seed_summary_daily_second_harmonic_val_report.csv", index = False)

# 4. Final Training on year 1 and year 2

In [11]:
df_train, df_test = build_features(
                                df_train = pd.read_excel("../Dataset/training.xlsx"), 
                                df_test = pd.read_excel("../Dataset/testing.xlsx")
                                   )

In [12]:
X_train = df_train[features]
y_train = df_train[target]

X_test = df_test[features]

In [13]:
print("Dataset sizes:")
print(f"Train (Year 1): {len(X_train):,}")
print(f"Validation (Year 2): {len(X_val):,}")
print(f"Test (Year 3): {len(X_test):,}")

Dataset sizes:
Train (Year 1): 17,544
Validation (Year 2): 8,760
Test (Year 3): 8,760


In [14]:
seeds = [0, 1, 2, 3, 42]

test_predictions = []

for seed in seeds:

    model = xgb.XGBRegressor(
        **best_params,
        objective="reg:squarederror",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    pred = model.predict(X_test)

    test_predictions.append(pred)

y_test_pred = np.mean(
    test_predictions,
    axis=0
)

In [15]:
pd.DataFrame(y_test_pred).to_csv("../report/daily_second_harmonic_test_prediction.csv", index = False)